In [30]:
import numpy as np
import pandas as pd
import math
import itertools
from scipy.optimize import minimize

# Manual Trading

In [3]:
payoffs= np.array([(10,1), (80,6), (37,3), (17,1),(31,2),(90,10),(50,4),(20,2),(73,4),(89,8)])

1st is always profitable
2nd is profitable iff p < 0.01*(0.2M-I)

Let's compute the following list : 0.01*(0.2M-I)

In [12]:
threshold = np.array([0]*len(payoffs), dtype=float)
print(threshold)
for i in range(len(payoffs)):
    mult, inhabs = payoffs[i]
    threshold[i] = 0.01*(0.2*mult-inhabs)
with np.printoptions(precision=3, suppress=True):
    print(threshold)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0.01  0.1   0.044 0.024 0.042 0.08  0.06  0.02  0.106 0.098]


In [10]:
avg_profit = np.median(threshold)
print(avg_profit)

0.052000000000000005


# If single Expedition 

maximizing the profit with the most adverse %
=> max 10000*M/(I+100)

In [39]:
def maximin1(p1):

    max_val = float('-inf')
    argmax = []
    for mult, inhabs in payoffs:
        val = (mult / (inhabs+100*p1) )* 10000
        if math.isclose(val, max_val):
            argmax.append((mult, inhabs))
        elif val > max_val:
            argmax = [(mult, inhabs)]
            max_val = val
    return (argmax, max_val)

In [40]:
maximin1(0)

([(73, 4)], 182500.0)

In [ ]:
def maximize2():
    max_val = float('-inf')
    max_mult1, max_inhabs1 = None, None
    max_mult2, max_inhabs2 = None, None

    for (mult1, inhabs1), (mult2, inhabs2) in itertools.combinations(payoffs, 2):
        def objective(p):
            p1, p2 = p
            return - (mult1 / (inhabs1 + 100 * p1) + mult2 / (inhabs2 + 100 * p2))  # négative to maximize

        constraints = [
            {'type': 'ineq', 'fun': lambda p: 1 - (p[0] + p[1])},  # p1 + p2 <= 1
            {'type': 'ineq', 'fun': lambda p: p[0]},               # p1 >= 0
            {'type': 'ineq', 'fun': lambda p: p[1]}                # p2 >= 0
        ]

        result = minimize(objective, x0=[0.5, 0.5], constraints=constraints, bounds=[(0,1), (0,1)])
        if result.success:
            val_temp = -result.fun  
            val = -50000 + val_temp * 10000
            if math.isclose(val, max_val):
                print("collision")
            if val > max_val:
                max_mult1, max_inhabs1 = mult1, inhabs1
                max_mult2, max_inhabs2 = mult2, inhabs2
                max_val = val

    return ((max_mult1, max_inhabs1), (max_mult2, max_inhabs2), max_val)

In [32]:
maximize2()

((17, 1), (73, 4), 302500.0)